# LangGraph — Agentic AI Workflows

In [ ]:
# Day 58 — LangGraph (Agentic AI)
# LangChain LCEL (Day 56) is a straight line (DAG): A -> B -> C.
# Real SRE/DevOps troubleshooting is a LOOP:
#   Diagnose -> Check Logs -> Run Tool -> Did it fix it? -> If NO, try another fix!
#
# LangGraph lets you build stateful, cyclical, multi-step AI agents as a StateGraph.
#
# Today we'll cover:
#   1. StateGraph & TypedDict state
#   2. Nodes (Python functions) & Edges (transitions)
#   3. Conditional routing & retry loops
#   4. Human-in-the-loop approval patterns for destructive DevOps actions
#
# Install: pip install langgraph langchain-openai

### 1. Defining State, Nodes & Conditional Edges

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END

# 1. Define the shared State passed between nodes
class IncidentState(TypedDict):
    pod_name: str
    status: str
    attempts: int
    diagnosis: str
    resolved: bool

# 2. Define Nodes (each takes state and returns partial state updates)
def diagnose_pod(state: IncidentState) -> dict:
    print(f"[Diagnose] Checking {state['pod_name']} (attempt {state['attempts'] + 1})")
    return {"diagnosis": "OOMKilled — memory limit too low", "attempts": state["attempts"] + 1}

def apply_fix(state: IncidentState) -> dict:
    print(f"[Fix] Increasing memory limit for {state['pod_name']}")
    # Simulate success on 2nd attempt
    is_ok = state["attempts"] >= 2
    return {"resolved": is_ok, "status": "Running" if is_ok else "CrashLoopBackOff"}

# 3. Conditional Router: loop or finish?
def route_after_fix(state: IncidentState) -> Literal["diagnose", "end"]:
    if state["resolved"] or state["attempts"] >= 3:
        return "end"
    return "diagnose"

# 4. Build & Compile the Graph
builder = StateGraph(IncidentState)
builder.add_node("diagnose", diagnose_pod)
builder.add_node("fix", apply_fix)

builder.set_entry_point("diagnose")
builder.add_edge("diagnose", "fix")
builder.add_conditional_edges("fix", route_after_fix, {"diagnose": "diagnose", "end": END})

graph = builder.compile()

result = graph.invoke({
    "pod_name": "ml-api-7d9f",
    "status": "CrashLoopBackOff",
    "attempts": 0,
    "diagnosis": "",
    "resolved": False
})
print("Final State:", result)

### 2. Human-in-the-Loop (HITL) for Safe DevOps Agents

In [ ]:
# In production DevOps, you NEVER let an LLM delete pods or scale prod
# without a safety check or human approval!
#
# LangGraph supports `interrupt_before=["execute_action"]` with a checkpointer:
#
#   from langgraph.checkpoint.memory import MemorySaver
#   graph = builder.compile(checkpointer=MemorySaver(), interrupt_before=["fix"])
#   # 1. Run until pause -> agent proposes the fix
#   # 2. Human SRE reviews and approves
#   # 3. Resume graph with graph.invoke(None, config=thread_config)

print("Always cap max `attempts` in cyclical graphs so an agent never loops forever!")

In [ ]:
# Practice Questions:
#
# 1. Why is LangGraph better than simple LCEL chains for autonomous SRE agents?
# 2. Why should every cyclical LangGraph workflow include an `attempts` counter?
# 3. How does `interrupt_before` enable safe human-in-the-loop approvals?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: Stateful Incident Responder
# Extend the graph above with a `verify_health` node that checks `/healthz`
# after `apply_fix` before marking `resolved = True`.

# Exercise 2: Multi-Agent DevOps Router
# Build a supervisor node that routes incoming tickets to either:
# `k8s_agent` (pod/deployment issues), `monitoring_agent` (CPU/drift), or `docs_agent` (RAG).

# Exercise 3: Human-in-the-Loop Deployment Approver
# Compile a StateGraph with `MemorySaver()` and `interrupt_before=["deploy_prod"]`
# so the workflow pauses for `input("Approve prod deploy? (y/n): ")`.